In [1]:
import torch
import torch.nn as nn
from spin_lattices import KagomeLattice, SquareLattice1Diag, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from loguru import logger
from slater_determinant import SlaterDeterminant
from pathlib import Path
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime


def sign_overlap(ground_state, predict_signs):
    probs = ground_state**2
    return torch.dot(ground_state, predict_signs * torch.abs(ground_state)) / probs.sum()

2023-04-20 17:36:15.302 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-04-20 17:36:15.305 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-04-20 17:36:15.355 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...


In [2]:

lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=1, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)

ground_state_np = np.real_if_close(system.get_ground_state_in_canonical_basis())
ground_state = torch.from_numpy(ground_state_np)

2023-04-20 17:36:19.019 | DEBUG    | heisenberg_hamiltonians:__init__:427 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-04-20 17:36:19.022 | DEBUG    | heisenberg_hamiltonians:__init__:448 - number_spins=24
2023-04-20 17:36:19.028 | DEBUG    | heisenberg_hamiltonians:__init__:458 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-20 17:36:19.221 | DEBUG    | heisenberg_hamiltonians:__init__:467 - Hilbert space dimension is 85662
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-20 17:36:19.364 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-True-1-1.pickle
2023-04-20 17:36:19.367 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -42.8245991763
2023-04-20 17:36:19.369 | DEBUG    | heisenberg

In [6]:
# with torch.no_grad():
#     det.f.copy_(
#         nn.Parameter(
#             torch.randn(system.number_spins, system.number_spins, dtype=torch.float64)
#             / np.sqrt(system.number_spins)
#         )
#     )

eps_train = 0.001
test_size = 10000
epochs = 1000
batch_size = 64
lr = 2e-2
initialization = "orthogonal"

dataset_seed = 2
np.random.seed(dataset_seed)
train_set_numpy = np.random.choice(
    len(system.canonical_basis.states),
    int(eps_train * len(system.canonical_basis.states)),
    replace=False,
    p=ground_state**2,
)

train_set = torch.from_numpy(train_set_numpy)
logger.debug(f"{len(train_set)=}")

target = (ground_state[train_set] > 0).double()

rest_set_np = np.setdiff1d(np.arange(len(system.canonical_basis.states)), train_set_numpy)
rest_probs = ground_state_np[rest_set_np] ** 2
rest_probs /= rest_probs.sum()
test_set = torch.from_numpy(np.random.choice(rest_set_np, test_size, replace=False))

for run in range(10):
    logger.debug(f"{run=}")
    writer = SummaryWriter(
        log_dir=f"experiments/2021_04_18/{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}_variable_init_run_{run}_dataset_seed_{dataset_seed}_eps_train_{eps_train}_batch_size_{batch_size}_lr_{lr}_initialization_{initialization}"
    )

    torch.manual_seed(run)
    det = SlaterDeterminant(
        system.canonical_basis, initialization=initialization, sign_cache_dir=Path("signs_cache")
    )

    n_batches = len(train_set) // batch_size

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(det.parameters(), lr=lr)

    epoch = 0
    logger.debug(f"{n_batches=}")
    for epoch in range(epochs):  # loop over the dataset multiple times
        i = None
        loss = None

        for i in range(n_batches):
            x = train_set[i * batch_size : (i + 1) * batch_size]
            y = target[i * batch_size : (i + 1) * batch_size]

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = torch.sigmoid(det(x))
            #        print(det(x))
            loss = criterion(outputs, y)
            loss.backward()
            #        print(det.f.grad.norm().item())

            optimizer.step()

        assert loss is not None
        overlap_train = sign_overlap(ground_state[train_set], torch.sign(det(train_set)))
        overlap_test = sign_overlap(ground_state[test_set], torch.sign(det(test_set)))
        # log.append(
        #     {
        #         "epoch": epoch,
        #         "loss": loss.item(),
        #         "overlap_train": overlap_train.item(),
        #         "overlap_test": overlap_test.item(),
        #     }
        # )
        # logger.debug(
        #     f"Epoch {epoch} loss: {loss.item():.4f} overlap_train: {overlap_train.item():.4f} "
        #     f"overlap_test: {overlap_test.item():.4f}"
        # )
        writer.add_scalar("Loss/train", loss.item(), epoch)
        writer.add_scalar("Overlap/train", overlap_train.item(), epoch)
        writer.add_scalar("Overlap/test", overlap_test.item(), epoch)

2023-04-20 17:45:06.821 | DEBUG    | __main__:<module>:26 - len(train_set)=2704
2023-04-20 17:45:07.104 | DEBUG    | __main__:<module>:36 - run=0
2023-04-20 17:45:09.683 | DEBUG    | slater_determinant:__init__:79 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-20 17:45:09.727 | DEBUG    | __main__:<module>:53 - n_batches=42
2023-04-20 17:46:41.609 | DEBUG    | __main__:<module>:36 - run=1
2023-04-20 17:46:43.847 | DEBUG    | slater_determinant:__init__:79 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-20 17:46:43.890 | DEBUG    | __main__:<module>:53 - n_batches=42
2023-04-20 17:48:12.705 | DEBUG    | __main__:<module>:36 - run=2
2023-04-20 17:48:15.229 | DEBUG    | slater_determinant:__init__:79 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-20 17:48:15.267 | DEBUG    | __main__:<module>:53 - n_batches=42
2023-04-20 17:49:41.012 | DEBUG    | __main__:<module>:36 -

In [ ]:
loss.item()

1.6168777209083929e-06

In [ ]:
overlap_train.item()

0.9741837977119633

In [ ]:
overlap_test.item()

0.16123035516625026